<a href="https://colab.research.google.com/github/JuanAntonioLeonOjeda/03MIAR---Algoritmos-de-Optimizacion/blob/main/Copia_de_Trabajo_Pr%C3%A1ctico_Algoritmos(mejorada).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Juan Antonio León Ojeda  <br>
Url: https://github.com/JuanAntonioLeonOjeda/03MIAR---Algoritmos-de-Optimizacion/tree/main/TRABAJO_PRACTICO<br>
Google Colab: https://colab.research.google.com/drive/17i9e51u3h6eMk9lQVqFDdGxHE9kPpins <br>
Problema:
>3. Configuración de Tribunales

Descripción del problema:

Se precisa configurar tribunales de evaluación para un grupo de 15 alumnos que desean
presentar su Trabajo Fin de Máster (TFM).
Cada tribunal está compuesto por tres profesores, cada uno desempeñando uno de los
siguientes roles: Presidente, Secretario o Vocal.
Los profesores han indicado su disponibilidad horaria para participar en los tribunales de
15h a 21h durante la semana del 15 al 19 de abril: <br>
Número de profesores : 10 <br>
Número de tribunales : 15 <br>
Hay 15 alumnos, por lo que se deben configurar 15 tribunales buscando la configuración
más equilibrada posible en cuanto a la cantidad de tribunales asignados a cada profesor, es
decir, evitando que un profesor tenga muchos tribunales y otros pocos.
Obviamente ningún profesor puede asistir a dos tribunales a la misma fecha/hora y no
puede ser convocado a un tribunal al que no tiene disponibilidad.









                                        

#Modelo

- ¿Como represento el espacio de soluciones?

In [2]:
# El espacio de soluciones será un array de diccionarios (podría usarse simplemente arrays anidados, pero los diccionarios proporcionarán una mayor claridad de cada asignación).
# El formato será como el que sigue:
'''
solucion = [
    {
        t: 0,      # tribunal
        d: 15,     # día
        h: 16,     # hora
        p: 'ABC',  # presidente
        s: 'DEF',  # secretario
        v: 'GHI'.  # vocal
    },
    ...
]
'''

"\nsolucion = [\n    {\n        t: 0,      # tribunal\n        d: 15,     # día\n        h: 16,     # hora\n        p: 'ABC',  # presidente\n        s: 'DEF',  # secretario\n        v: 'GHI'.  # vocal\n    },\n    ...\n]\n"

- ¿Cual es la función objetivo?

El objetivo principal del modelo tiene que ser alcanzar un reparto equilibrado de asignaciones de tribunales para cada profesor.
Para ello, se calculará la desviación estándar para medir la dispersión de los valores (cantidad de asignaciones) respecto a la media.
La función objetivo tiene que intentar que esta desviación estándar sea la menor posible para cada profesor, a través de la siguiente fórmula

σ = √[Σ(xi - x̄)² / n]

Donde:
xi: número de tribulaes asignados al profesor i
x̄: media de asignaciones
n: total de profesores

- ¿Como implemento las restricciones?

In [3]:
# Las restricciones serán un conjunto de arrays y de diccionarios. Básicamente se usarán arrays para listar los elementos por los que hay que iterar (profesores, días y horas) y diccionarios para hacer consultas (disponibilidad y roles de cada profesro)

profesores = ['RRD', 'QYV', 'LHL', 'HLC', 'MSB', 'PMQ', 'QWF', 'EBB', 'IOE', 'IOA']
dias = [15, 16, 17, 18, 19]
horas = [15, 16, 17, 18, 19, 20, 21]
disponibilidad = {
    'RRD': { # Revisado
        (15, 15): 0, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 1,
        (16, 15): 1, (16, 16): 0, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 0, (17, 18): 0, (17, 19): 1, (17, 20): 0, (17, 21): 1,
        (18, 15): 0, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 0, (19, 21): 0,
    },
    'QYV': { # Revisado
        (15, 15): 1, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 0, (15, 21): 0,
        (16, 15): 0, (16, 16): 1, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 0, (16, 21): 0,
        (17, 15): 1, (17, 16): 0, (17, 17): 0, (17, 18): 1, (17, 19): 1, (17, 20): 1, (17, 21): 0,
        (18, 15): 1, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 1, (19, 21): 1,
    },
    'LHL': { #Revisado
        (15, 15): 0, (15, 16): 0, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 1,
        (16, 15): 1, (16, 16): 1, (16, 17): 1, (16, 18): 0, (16, 19): 0, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 1, (17, 18): 1, (17, 19): 1, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 0, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 0, (18, 21): 1,
        (19, 15): 0, (19, 16): 1, (19, 17): 1, (19, 18): 0, (19, 19): 1, (19, 20): 0, (19, 21): 1,
    },
    'HLC': { # Revisado
        (15, 15): 1, (15, 16): 0, (15, 17): 1, (15, 18): 0, (15, 19): 1, (15, 20): 1, (15, 21): 0,
        (16, 15): 1, (16, 16): 0, (16, 17): 0, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 0, (17, 16): 0, (17, 17): 1, (17, 18): 1, (17, 19): 1, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 0, (18, 17): 1, (18, 18): 1, (18, 19): 0, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 1, (19, 21): 0,
    },
    'MSB': { # Revisado
        (15, 15): 1, (15, 16): 1, (15, 17): 0, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 1,
        (16, 15): 1, (16, 16): 1, (16, 17): 1, (16, 18): 0, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 0, (17, 17): 1, (17, 18): 1, (17, 19): 0, (17, 20): 1, (17, 21): 1,
        (18, 15): 0, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 0, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 0, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 1, (19, 21): 0,
    },
    'PMQ': { # Revisado
        (15, 15): 1, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 1, (15, 20): 0, (15, 21): 0,
        (16, 15): 1, (16, 16): 1, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 0, (17, 18): 0, (17, 19): 1, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 1, (18, 17): 1, (18, 18): 0, (18, 19): 0, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 0, (19, 19): 1, (19, 20): 0, (19, 21): 1,
    },
    'QWF': { # Revisado
        (15, 15): 0, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 1, (15, 20): 1, (15, 21): 1,
        (16, 15): 1, (16, 16): 1, (16, 17): 0, (16, 18): 1, (16, 19): 1, (16, 20): 0, (16, 21): 1,
        (17, 15): 0, (17, 16): 0, (17, 17): 1, (17, 18): 1, (17, 19): 0, (17, 20): 0, (17, 21): 1,
        (18, 15): 1, (18, 16): 0, (18, 17): 0, (18, 18): 0, (18, 19): 0, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 0, (19, 21): 1,
    },
    'EBB': { # Revisado
        (15, 15): 1, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 1, (15, 20): 0, (15, 21): 0,
        (16, 15): 1, (16, 16): 1, (16, 17): 0, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 0,
        (17, 15): 1, (17, 16): 1, (17, 17): 1, (17, 18): 0, (17, 19): 0, (17, 20): 1, (17, 21): 1,
        (18, 15): 0, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 0, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 0, (19, 20): 1, (19, 21): 0,
    },
    'IOE': { # Revisado
        (15, 15): 1, (15, 16): 0, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 0,
        (16, 15): 0, (16, 16): 1, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 0, (17, 18): 0, (17, 19): 0, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 0,
        (19, 15): 1, (19, 16): 0, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 1, (19, 21): 1,
    },
    'IOA': { # Revisado
        (15, 15): 1, (15, 16): 1, (15, 17): 0, (15, 18): 1, (15, 19): 1, (15, 20): 0, (15, 21): 1,
        (16, 15): 1, (16, 16): 0, (16, 17): 0, (16, 18): 0, (16, 19): 0, (16, 20): 0, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 0, (17, 18): 0, (17, 19): 1, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 0, (18, 17): 0, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 0, (19, 19): 0, (19, 20): 0, (19, 21): 1,
    },
}

roles = {
  'RRD': ['P', 'S', 'V'],
  'QYV': ['P', 'S', 'V'],
  'LHL': ['P', 'V'],
  'HLC': ['S', 'V'],
  'MSB': ['P', 'S', 'V'],
  'PMQ': ['P', 'S', 'V'],
  'QWF': ['S', 'V'],
  'EBB': ['S', 'V'],
  'IOE': ['P', 'S', 'V'],
  'IOA': ['P', 'S', 'V'],
}

#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones

In [4]:
#Respuesta


#Diseño
- ¿Que técnica utilizo? ¿Por qué?

He empleado la técnica de Ramificación y Poda, ya que se trata una función objetivo para optimización y con restricciones específicas. Además, el problema trabaja solo con números enteros (1 y 0), lo cual encaja con esta técnica.
He procurado también usar la técnica de 'Divide y vencerás', aplicando recursividad para evitar bucles innecesarios.

# Resolución

In [24]:
# Preparar datos
# Listas de profesores aptos para cada rol, para no tener que hacer la comprobación en cada iteración
presidentes = [ p for p in profesores if 'P' in roles[p] ]
secretarios = [ p for p in profesores if 'S' in roles[p] ]
vocales = [ p for p in profesores if 'V' in roles[p] ]

# Todos los turnos disponibles para consultar disponibilidad más tarde
turnos = [
  (15, 15), (15, 16), (15, 17), (15, 18), (15, 19), (15, 20), (15, 21), # día 15
  (16, 15), (16, 16), (16, 17), (16, 18), (16, 19), (16, 20), (16, 21), # día 16
  (17, 15), (17, 16), (17, 17), (17, 18), (17, 19), (17, 20), (17, 21), # día 17
  (18, 15), (18, 16), (18, 17), (18, 18), (18, 19), (18, 20), (18, 21), # día 18
  (19, 15), (19, 16), (19, 17), (19, 18), (19, 19), (19, 20), (19, 21)  # día 19
]

# Prefiltrado previo, donde hacemos una lista de profesores disponibles para cada turno
def revisar_disponibilidad():
  resultado = {}
  for turno in turnos:
      resultado[turno] = [ p for p in profesores if disponibilidad[p][turno] ]
  return resultado

# Crear diccionario con los profesores disponibes para cada día y hora
disp = revisar_disponibilidad()

#---------------------------------------------------------------------
#Inicialización variables
mejor_solucion = None
cota_superior = float('inf') # desviación típica inicial de las cargas de trabajo de los profesores
solucion_actual = []
tribunales_ocupados = {}
carga_profesores = {p: 0 for p in profesores}
max = 15 # número total de tribunales a asignar

# Ordenar los turnos por cantidad de profesores disponibles, y seleccionar los 15 primeros turnos
turnos_por_disponibilidad = []
for slot in turnos:
  dia, hora = slot
  disponibles = len(disp[(dia, hora)])
  turnos_por_disponibilidad.append((slot, disponibles))

turnos_por_disponibilidad.sort(key=lambda x: x[1], reverse=True) # De mayor a menor

turnos_seleccionados = [turno for turno, _ in turnos_por_disponibilidad[:max]] # 15 turnos con mayor disponibilidad de profesores (usaremos para determinar la solución inicial)

#---------------------------------------------------------------------
# Funciones usadas durante el ejercicio

# Función Objetivo
def calcular_desv_tipica(cargas = carga_profesores):
  total_profesores = len(cargas)
  valores = list(cargas.values())
  media = sum(valores) / total_profesores
  varianza = sum((v - media)**2 for v in valores) / total_profesores   # σ = √[Σ(xi - x̄)² / n]
  return varianza ** 0.5

def esta_ocupado(p, d, h):
  return (d, h) in tribunales_ocupados and p in tribunales_ocupados[(d, h)]

def asignar_tribunal(n, dia, hora, pres, sec, voc):
  solucion_actual.append({'tribunal': n + 1, 'dia': dia, 'hora': hora, 'P': pres, 'S': sec, 'V': voc})
  if (dia, hora) not in tribunales_ocupados:
    tribunales_ocupados[(dia, hora)] = []

  # Marcar qué profesores ocuparán el turno y sumar la asignación a cada profesor
  for prof in [pres, sec, voc]:
    tribunales_ocupados[(dia, hora)].append(prof)
    carga_profesores[prof] += 1

def deshacer_tribunal():
  if not solucion_actual:
    return
  t = solucion_actual.pop()

  # Desmarcar los profesores que ocupan el turno, y restar asignación
  for prof in [t['P'], t['S'], t['V']]:
    carga_profesores[prof] -= 1
    tribunales_ocupados[(t['dia'], t['hora'])].remove(prof)

  if not tribunales_ocupados[(t['dia'], t['hora'])]:
    del tribunales_ocupados[(t['dia'], t['hora'])]

def calcular_cota_inferior(idx):
  tribunales_asignados = idx
  tribunales_restantes = max - tribunales_asignados

  if tribunales_restantes == 0:
    return calcular_desv_tipica()

  # Simular la distribución de los tribunales restantes de forma óptima
  asignaciones_restantes = tribunales_restantes * 3
  cargas_simuladas = carga_profesores.copy() # copy para no modificar el original

  for _ in range(asignaciones_restantes):
    profesor_menos_cargado = min(cargas_simuladas, key=cargas_simuladas.get)
    cargas_simuladas[profesor_menos_cargado] += 1

  # Devolver la desviación típica de esta simulación para ver si merece la pena explorarla
  return calcular_desv_tipica(cargas_simuladas)

def generar_tribunales(idx_turno):
  global mejor_solucion, cota_superior

  # Condición de parada
  if idx_turno >= max:
    reparto = calcular_desv_tipica()

    # Actualizar cota superior si encontramos mejor
    if reparto < cota_superior:
      cota_superior = reparto
      mejor_solucion = [t.copy() for t in solucion_actual]
    return

  # Creamos una rama y vemos si merece la pena explorar
  cota_inferior = calcular_cota_inferior(idx_turno)
  if cota_inferior >= cota_superior:
    # Podamos las cotas inferiores que no puedan mejorar la solución
    return

  #Si lleguamos aquí, exploramos la rama
  dia, hora = turnos_seleccionados[idx_turno]
  disponibles = set(disp[(dia, hora)])
  libres = {p for p in disponibles if not esta_ocupado(p, dia, hora)}

  # Verificar que haya al menos un profesor para cada rol. Ordenamos por carga
  prof_p = sorted(libres & set(presidentes), key=lambda p: carga_profesores[p])
  prof_s = sorted(libres & set(secretarios), key=lambda p: carga_profesores[p])
  prof_v = sorted(libres & set(vocales), key=lambda p: carga_profesores[p])

  if prof_p and prof_s and prof_v:
    # Asegurar que haya al menos un profesor distinto del resto para cada rol
    for p in prof_p:
      for s in prof_s:
        if s == p:
          continue
        for v in prof_v:
          if v in [p, s]:
            continue

          asignar_tribunal(idx_turno, dia, hora, p, s, v)
          generar_tribunales(idx_turno + 1)
          deshacer_tribunal() # Resetear asignación para explorar la siguiente itereación

# Inicio del prodecimiento, empezando en el índice 0 (1er tribunal)
generar_tribunales(0)

print("\n" + "="*60)
print("RESULTADO FINAL")
print("="*60)
if mejor_solucion:
  print(f"Reparto final: {cota_superior:.4f}")

  cargas_finales = {p: 0 for p in profesores}
  for t in mejor_solucion:
    cargas_finales[t['P']] += 1
    cargas_finales[t['S']] += 1
    cargas_finales[t['V']] += 1

  print("\nCargas:")
  for prof, carga in sorted(cargas_finales.items(), key=lambda x: x[1], reverse=True):
    print(f"  {prof}: {carga}")

  print("\nTribunales:")
  for t in sorted(mejor_solucion, key=lambda x: x['tribunal']):
    print(f"  T{t['tribunal']}: Día {t['dia']}, {t['hora']}:00 - P:{t['P']}, S:{t['S']}, V:{t['V']}")
else:
  print("No se encontró solución")

{(19, 17): ['PMQ']}
{(19, 17): ['PMQ', 'HLC']}
{(19, 17): ['PMQ', 'HLC', 'EBB']}
{(19, 17): ['PMQ', 'HLC', 'EBB'], (15, 18): ['RRD']}
{(19, 17): ['PMQ', 'HLC', 'EBB'], (15, 18): ['RRD', 'MSB']}
{(19, 17): ['PMQ', 'HLC', 'EBB'], (15, 18): ['RRD', 'MSB', 'LHL']}
{(19, 17): ['PMQ', 'HLC', 'EBB'], (15, 18): ['RRD', 'MSB', 'LHL'], (17, 21): ['IOA']}
{(19, 17): ['PMQ', 'HLC', 'EBB'], (15, 18): ['RRD', 'MSB', 'LHL'], (17, 21): ['IOA', 'QWF']}
{(19, 17): ['PMQ', 'HLC', 'EBB'], (15, 18): ['RRD', 'MSB', 'LHL'], (17, 21): ['IOA', 'QWF', 'IOE']}
{(19, 17): ['PMQ', 'HLC', 'EBB'], (15, 18): ['RRD', 'MSB', 'LHL'], (17, 21): ['IOA', 'QWF', 'IOE'], (18, 20): ['QYV']}
{(19, 17): ['PMQ', 'HLC', 'EBB'], (15, 18): ['RRD', 'MSB', 'LHL'], (17, 21): ['IOA', 'QWF', 'IOE'], (18, 20): ['QYV', 'PMQ']}
{(19, 17): ['PMQ', 'HLC', 'EBB'], (15, 18): ['RRD', 'MSB', 'LHL'], (17, 21): ['IOA', 'QWF', 'IOE'], (18, 20): ['QYV', 'PMQ', 'EBB']}
{(19, 17): ['PMQ', 'HLC', 'EBB'], (15, 18): ['RRD', 'MSB', 'LHL'], (17, 21): ['IOA

# Uso de IA

La IA se usó inicialmente para generar el diccionario de disponibilidad de profesores, ya que era muy extenso. Necesité consultar qué concepto matemático usar para medir el reparto de cargas para poder plantear la solución. También se usó para consultar si el planteamiento era cohoerente, y constultar posibles cambios de planteamiento y si mejoraba su eficiencia. Dejo a continuación conversaciones con la IA.

Porfa, crea un diccionario con cada profesor y la disponibilidad que tienen para los días y horas especificadas, con un formato como este:

disponibilidad = {
  'RRD': {
    (15, 15): 0, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 1, etc
...
  }
}
adjunto imagen para que veas la información

In [ ]:
'''
Viewing the availability matrix image to extract the data for all professors
Perfecto, veo la matriz completa. Te creo el diccionario con todos los datos:

disponibilidad = {
    'RRD': {
        (15, 15): 0, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 1,
        (16, 15): 1, (16, 16): 0, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 0, (17, 18): 0, (17, 19): 1, (17, 20): 0, (17, 21): 1,
        (18, 15): 0, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 0, (19, 21): 0,
    },
    'QYV': {
        (15, 15): 1, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 0, (15, 21): 0,
        (16, 15): 0, (16, 16): 1, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 0, (16, 21): 0,
        (17, 15): 1, (17, 16): 1, (17, 17): 0, (17, 18): 0, (17, 19): 1, (17, 20): 1, (17, 21): 1,
        (18, 15): 0, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 1, (19, 21): 1,
    },
    'LJL': {
        (15, 15): 1, (15, 16): 0, (15, 17): 0, (15, 18): 1, (15, 19): 1, (15, 20): 0, (15, 21): 1,
        (16, 15): 1, (16, 16): 1, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 0, (16, 21): 0,
        (17, 15): 1, (17, 16): 1, (17, 17): 1, (17, 18): 1, (17, 19): 1, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 1, (18, 17): 1, (18, 18): 0, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 0, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 0, (19, 20): 1, (19, 21): 0,
    },
    'HLC': {
        (15, 15): 1, (15, 16): 0, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 1,
        (16, 15): 0, (16, 16): 1, (16, 17): 1, (16, 18): 0, (16, 19): 0, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 0, (17, 17): 0, (17, 18): 1, (17, 19): 1, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 1, (18, 17): 1, (18, 18): 0, (18, 19): 1, (18, 20): 0, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 1, (19, 21): 0,
    },
    'MSB': {
        (15, 15): 1, (15, 16): 1, (15, 17): 0, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 1,
        (16, 15): 1, (16, 16): 1, (16, 17): 1, (16, 18): 0, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 1, (17, 18): 0, (17, 19): 1, (17, 20): 1, (17, 21): 1,
        (18, 15): 0, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 0, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 0, (19, 19): 1, (19, 20): 1, (19, 21): 0,
    },
    'PMQ': {
        (15, 15): 1, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 1, (15, 20): 0, (15, 21): 0,
        (16, 15): 1, (16, 16): 1, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 1, (17, 18): 0, (17, 19): 0, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 0, (18, 21): 0,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 0, (19, 20): 1, (19, 21): 0,
    },
    'QWF': {
        (15, 15): 0, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 1, (15, 20): 1, (15, 21): 1,
        (16, 15): 1, (16, 16): 1, (16, 17): 0, (16, 18): 1, (16, 19): 1, (16, 20): 0, (16, 21): 1,
        (17, 15): 0, (17, 16): 0, (17, 17): 1, (17, 18): 1, (17, 19): 1, (17, 20): 0, (17, 21): 0,
        (18, 15): 1, (18, 16): 1, (18, 17): 1, (18, 18): 0, (18, 19): 0, (18, 20): 0, (18, 21): 0,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 0, (19, 21): 1,
    },
    'EBB': {
        (15, 15): 1, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 1, (15, 20): 0, (15, 21): 0,
        (16, 15): 1, (16, 16): 1, (16, 17): 0, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 0, (17, 16): 1, (17, 17): 1, (17, 18): 1, (17, 19): 0, (17, 20): 0, (17, 21): 1,
        (18, 15): 1, (18, 16): 1, (18, 17): 0, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 0, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 1, (19, 21): 0,
    },
    'IOE': {
        (15, 15): 1, (15, 16): 0, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 0,
        (16, 15): 0, (16, 16): 1, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 1, (17, 18): 0, (17, 19): 0, (17, 20): 0, (17, 21): 1,
        (18, 15): 1, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 0, (19, 17): 1, (19, 18): 0, (19, 19): 1, (19, 20): 1, (19, 21): 1,
    },
    'IOA': {
        (15, 15): 1, (15, 16): 1, (15, 17): 0, (15, 18): 1, (15, 19): 1, (15, 20): 0, (15, 21): 1,
        (16, 15): 1, (16, 16): 0, (16, 17): 0, (16, 18): 0, (16, 19): 0, (16, 20): 0, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 1, (17, 18): 0, (17, 19): 0, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 0, (18, 17): 0, (18, 18): 0, (18, 19): 0, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 0, (19, 20): 0, (19, 21): 0,
    },
}

#Listo! Ahora puedes usar el diccionario así:
# Verificar si RRD está disponible el día 15 a las 16h
if disponibilidad['RRD'][(15, 16)] == 1:
    print("RRD disponible")

# Verificar si QYV está disponible el día 18 a las 20h
if disponibilidad['QYV'][(18, 20)] == 1:
    print("QYV disponible")
'''

tengo un grupo de 15 profesores a los que tengo que asignar un número total de tribunales.  se deben configurar 15 tribunales buscando la configuración
más equilibrada posible en cuanto a la cantidad de tribunales asignados a cada profesor, es
decir, evitando que un profesor tenga muchos tribunales y otros pocos.

cómo puedo calcular matemáticamente este reparto de carga?

In [ ]:
'''Para calcular el reparto más equilibrado de 15 tribunales entre 15 profesores, puedes usar este enfoque matemático:
Cálculo básico
Carga promedio = Total de tribunales / Total de profesores = 15/15 = 1 tribunal por profesor
En este caso específico, la solución es directa: cada profesor debería tener exactamente 1 tribunal asignado.
Fórmula general (para cualquier caso)
Para casos donde los números no dividen exactamente, usa:

Carga base: q = ⌊n/p⌋ (división entera)

donde n = número de tribunales, p = número de profesores


Profesores con carga extra: r = n mod p (resto de la división)

r profesores tendrán (q+1) tribunales
(p-r) profesores tendrán q tribunales


Desviación estándar (para medir equilibrio):

   σ = √[Σ(xi - x̄)² / p]
donde xi es la carga de cada profesor y x̄ es la media'''

'''def generar_tribunal(n, max, profesores, dias, horas, roles, solucion):
  solucion.append({
      'tribunal': n
  })
  if n == max:
    return solucion
  return generar_tribunal(n+1, max, profesores, dias, horas, roles, solucion)

def start(max, profesores, dias, horas, roles):
  solucion = []
  
  return generar_tribunal(1, max, profesores, dias, horas, roles, solucion)


print(start(15, profesores, dias, horas, roles))'''

cómo lo ves para empezar a plantear la solución?

In [ ]:
'''Análisis de tu Enfoque Inicial
Evaluación: 3/10 ⚠️
Tu código tiene la idea correcta (recursión para generar tribunales), pero tiene problemas estructurales importantes.
...
'''


coño te estoy diciendo para EMPEZAR ,  solo quiero hacerme una idea de si la recursividad está bien planteada

In [ ]:
'''¡Ah, vale! Solo para la Recursividad Base
Respuesta Directa: SÍ, está BIEN planteada ✅
Tu recursividad básica es correcta para empezar:
pythondef generar_tribunal(n, max, ...):
    solucion.append({'tribunal': n})

    if n == max:                          # ← Caso base correcto
        return solucion

    return generar_tribunal(n+1, ...)     # ← Llamada recursiva correcta

Lo que está BIEN:
✅ Caso base correcto: if n == max: return
✅ Recursión avanza: n+1 progresa hacia el final
✅ Retorna resultado: return solucion
✅ Estructura clara: Fácil de entender

Pequeño ajuste para que funcione mejor:
pythondef generar_tribunal(n, max, profesores, dias, horas, roles, solucion):
    # Caso base PRIMERO (mejor práctica)
    if n > max:  # ← Cambio: > en vez de ==
        return solucion

    solucion.append({
        'tribunal': n,
        # Aquí luego añadirás: 'dia': ..., 'hora': ..., etc.
    })

    return generar_tribunal(n+1, max, profesores, dias, horas, roles, solucion)'''

podría ser más eficiente mirar primero días y horas, y hacer listado de los profesores disponbiels por rol que ir profe por profe para cada rol?

In [ ]:
'''
¿Invertir la Lógica: Días/Horas → Profesores?Respuesta Directa: SÍ, podría ser MÁS eficiente ✅
Análisis de Eficiencia
Costo de Pre-cálculo (una sola vez):
python# Por cada slot (15 slots):
# - Verificar 7 candidatos_P
# - Verificar 9 candidatos_S
# - Verificar 10 candidatos_V

# Total: 15 × (7 + 9 + 10) = 15 × 26 = 390 verificaciones
# Se hace UNA SOLA VEZ al inicio
Costo por Tribunal:
python# Enfoque A (profesor → día):
# 700 verificaciones por tribunal × 15 tribunales = 10,500 verificaciones

# Enfoque B (día → profesor):
# 0 verificaciones por tribunal (ya pre-calculado)
# Solo costo inicial: 390 verificaciones

# GANANCIA: 10,500 → 390 = 96.3% menos verificaciones
'''

usar sets sería más eficiente que usar for?

In [ ]:
'''
¿Sets vs For Loops?Respuesta Directa: Depende de QUÉ estés haciendo ⚠️No es "Sets vs For" - Ambos se Usan Juntospython# MALENTENDIDO COMÚN:
# "¿Uso sets O uso for?"

# REALIDAD:
# Sets y for loops hacen cosas DIFERENTES
# Normalmente usas AMBOS juntosCaso 1: Intersección (Encontrar Elementos Comunes)Con Listas + For Loop:pythondisponibles = ['QYV', 'HLC', 'MSB', 'PMQ', 'EBB', 'IOE', 'IOA']
presidentes = ['RRD', 'QYV', 'LJL', 'MSB', 'PMQ', 'IOE', 'IOA']

# Encontrar quién está disponible Y puede ser presidente
candidatos_P = []
for p in disponibles:           # 7 iteraciones
    if p in presidentes:        # Cada "in" busca en lista → O(7)
        candidatos_P.append(p)

# Complejidad: O(7 × 7) = O(49) operacionesCon Sets (sin for explícito):pythondisponibles_set = set(disponibles)
presidentes_set = set(presidentes)

# Intersección directa
candidatos_P = disponibles_set & presidentes_set

# Complejidad: O(min(7, 7)) = O(7) operaciones
# 7x MÁS RÁPIDOResultado: Sets ganan ✅Caso 2: Iteración Simple (Procesar Cada Elemento)Con For Loop:pythondisponibles = ['QYV', 'HLC', 'MSB', 'PMQ', 'EBB', 'IOE', 'IOA']

# Imprimir cada profesor
for prof in disponibles:
    print(prof)

# Complejidad: O(n)Con Set:pythondisponibles_set = {'QYV', 'HLC', 'MSB', 'PMQ', 'EBB', 'IOE', 'IOA'}

# Imprimir cada profesor
for prof in disponibles_set:  # ← Sigues usando for!
    print(prof)

# Complejidad: O(n)Resultado: IGUAL (no hay ventaja) 🟰Caso 3: Búsqueda (Verificar si Existe)Con Lista:pythonpresidentes = ['RRD', 'QYV', 'LJL', 'MSB', 'PMQ', 'IOE', 'IOA']

# Verificar si 'MSB' puede ser presidente
if 'MSB' in presidentes:  # Busca secuencialmente
    print("Puede")

# Complejidad: O(n) - puede recorrer toda la lista
# En promedio: O(n/2)Con Set:pythonpresidentes_set = {'RRD', 'QYV', 'LJL', 'MSB', 'PMQ', 'IOE', 'IOA'}

# Verificar si 'MSB' puede ser presidente
if 'MSB' in presidentes_set:  # Lookup en hash table
    print("Puede")

# Complejidad: O(1) - directoResultado: Set gana ✅

Tabla Comparativa
OperaciónLista + ForSetGanadorIntersecciónO(n × m)O(min(n, m))Set 🏆Búsqueda (in)O(n)O(1)Set 🏆Iteración simpleO(n)O(n)Empate 🟰Acceso por índiceO(1)❌ No soportaLista 🏆Mantener orden✅ Sí❌ NoLista 🏆Eliminar duplicadosRequiere código extraAutomáticoSet 🏆
'''